# 03 · Selectivity Explanation: A vs B vs C
Central analysis notebook. Compares three explanation strategies for the
D2/5HT2A atypicality ratio using SHAP in a common fingerprint space.

| Approach | Model | Explanation target |
|---|---|---|
| **A** | Direct selectivity model (rf_sel) | expl(f(Δ)) |
| **B** | Two independent models subtracted | expl(f(5HT2A)) − expl(f(D2)) |
| **C** | Multi-output model subtracted | expl(f_mt(5HT2A)) − expl(f_mt(D2)) |

**Input:** `data.pkl`, `models.pkl`

In [ ]:
import sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from utils import (build_feature_matrix, expand_shap_to_fp,
                   draw_fragment_grid, SEED, N_FP)
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr, pearsonr
import shap

np.random.seed(SEED)
print("Imports OK.")

## 1. Load data & models

In [ ]:
with open('data.pkl','rb') as f:  data   = pickle.load(f)
with open('models.pkl','rb') as f: _saved = pickle.load(f)

d2 = data['d2']; sht = data['sht']; merged = data['merged']
X_d2 = data['X_d2']; X_sht = data['X_sht']; X_ov = data['X_ov']

from utils import patch_xgb

# ── Handle both pkl formats ────────────────────────────────────────────────────
# Old (model_tuning.ipynb): keys 'best_models', 'selectors', 'scalers'
# New (02_models.ipynb):    keys 'rf_d2', 'rf_5ht2a', 'rf_sel', 'rf_mt', ...
if 'best_models' in _saved:
    _bm      = _saved['best_models']
    rf_d2    = patch_xgb(_bm['D2']['model'])
    rf_5ht2a = patch_xgb(_bm['5HT2A']['model'])
    rf_sel   = patch_xgb(_bm['Delta']['model'])
    rf_mt    = None   # not saved in old format; trained fresh in next cell
    _r2_d2   = _bm['D2']['test_r2']
    _r2_sht  = _bm['5HT2A']['test_r2']
    _r2_del  = _bm['Delta']['test_r2']
    _nm_d2   = _bm['D2']['model_name']
    _nm_sht  = _bm['5HT2A']['model_name']
    _nm_del  = _bm['Delta']['model_name']
else:
    rf_d2    = patch_xgb(_saved['rf_d2']['model'])
    rf_5ht2a = patch_xgb(_saved['rf_5ht2a']['model'])
    rf_sel   = patch_xgb(_saved['rf_sel']['model'])
    rf_mt    = _saved.get('rf_mt', {}).get('model', None)
    _r2_d2   = _saved['rf_d2']['test_r2']
    _r2_sht  = _saved['rf_5ht2a']['test_r2']
    _r2_del  = _saved['rf_sel']['test_r2']
    _nm_d2   = _saved['rf_d2']['name']
    _nm_sht  = _saved['rf_5ht2a']['name']
    _nm_del  = _saved['rf_sel']['name']

sel = _saved['selectors']
sc  = _saved['scalers']

# If multi-task model is missing (old format), train it now
if rf_mt is None:
    print("Multi-output RF not in pkl — training fresh (this takes ~30s)...")
    from sklearn.ensemble import RandomForestRegressor
    from utils import scaffold_split
    y_d2_ov  = merged['pChEMBL_D2'].values
    y_sht_ov = merged['pChEMBL_5HT2A'].values
    from utils import build_feature_matrix
    X_ov_feat_tmp = build_feature_matrix(X_ov, merged['curated_smiles'].tolist(),
                                         sel['overlap'], sc['overlap'])
    if 'splits' in _saved:
        tr_ov = _saved['splits']['tr_ov']
        te_ov = _saved['splits']['te_ov']
    else:
        tr_ov, te_ov = scaffold_split(merged, seed=42)
    Y_mt  = np.column_stack([y_d2_ov, y_sht_ov])
    rf_mt = RandomForestRegressor(n_estimators=300, max_depth=15,
                                   min_samples_leaf=3, random_state=42, n_jobs=-1)
    rf_mt.fit(X_ov_feat_tmp[tr_ov], Y_mt[tr_ov])
    print("Multi-output RF trained.")

y_del     = merged['delta'].values
smiles_ov = merged['curated_smiles'].tolist()

print(f"D2: {_nm_d2} R2={_r2_d2:.3f} | 5HT2A: {_nm_sht} R2={_r2_sht:.3f} | Delta: {_nm_del} R2={_r2_del:.3f}")

## 2. Build per-model feature matrices & SHAP subsample
Each model has its own variance-filtered feature space. We build separate input matrices, run SHAP in each space, then expand all attributions back to the common 2048-bit FP space for fair comparison.

In [ ]:
N_SHAP   = 1000
rng      = np.random.RandomState(SEED)
idx_shap = np.sort(rng.choice(len(X_ov), size=min(N_SHAP, len(X_ov)), replace=False))

fps_shap    = X_ov[idx_shap]
smiles_shap = [smiles_ov[i] for i in idx_shap]
merged_shap = merged.iloc[idx_shap].reset_index(drop=True)
y_shap      = y_del[idx_shap]

# Per-model feature matrices
X_sh_d2  = build_feature_matrix(fps_shap, smiles_shap, sel['D2'],    sc['D2'])
X_sh_sht = build_feature_matrix(fps_shap, smiles_shap, sel['5HT2A'], sc['5HT2A'])
X_sh_ov  = build_feature_matrix(fps_shap, smiles_shap, sel['overlap'],sc['overlap'])

print(f"SHAP subsample: {len(idx_shap)} compounds")
print(f"Feature dims: D2={X_sh_d2.shape[1]}  5HT2A={X_sh_sht.shape[1]}  OV={X_sh_ov.shape[1]}")

In [ ]:
print("Computing SHAP values (each model in its own feature space)...")
kw = dict(feature_perturbation='tree_path_dependent')

sv_d2    = shap.TreeExplainer(rf_d2,    **kw).shap_values(X_sh_d2)
print("  D2 done")
sv_5ht2a = shap.TreeExplainer(rf_5ht2a, **kw).shap_values(X_sh_sht)
print("  5HT2A done")
sv_sel   = shap.TreeExplainer(rf_sel,   **kw).shap_values(X_sh_ov)
print("  Selectivity done")
sv_mt    = shap.TreeExplainer(rf_mt,    **kw).shap_values(X_sh_ov)
print("  Multi-output done")

# Unpack multi-output
if isinstance(sv_mt, list):
    sv_mt_d2, sv_mt_sht = sv_mt[0], sv_mt[1]
elif sv_mt.ndim == 3:
    sv_mt_d2, sv_mt_sht = sv_mt[:,:,0], sv_mt[:,:,1]

# Expand to common 2048-bit FP space
shap_A = expand_shap_to_fp(sv_sel,    sel['overlap'])
shap_B = expand_shap_to_fp(sv_5ht2a,  sel['5HT2A']) - expand_shap_to_fp(sv_d2,     sel['D2'])
shap_C = expand_shap_to_fp(sv_mt_sht, sel['overlap'])- expand_shap_to_fp(sv_mt_d2,  sel['overlap'])

imp_A = np.abs(shap_A).mean(axis=0)
imp_B = np.abs(shap_B).mean(axis=0)
imp_C = np.abs(shap_C).mean(axis=0)

print(f"Common SHAP space: {shap_A.shape}  (N x {N_FP} FP bits)")

## 3. Three-way comparison metrics

In [ ]:
pairs = [('A','B'), ('A','C'), ('B','C')]
shaps = {'A': shap_A, 'B': shap_B, 'C': shap_C}
imps  = {'A': imp_A,  'B': imp_B,  'C': imp_C}

results = {}
print(f"{'Pair':>8} | {'Spearman':>10} | {'Pearson':>8} | {'Top50':>8} | {'cos_sim':>10} | {'%>0.8':>7}")
print("-"*60)
for p1, p2 in pairs:
    rho,  _ = spearmanr(imps[p1], imps[p2])
    pear, _ = pearsonr(imps[p1],  imps[p2])
    ol50    = len(set(np.argsort(imps[p1])[::-1][:50].tolist()) &
                  set(np.argsort(imps[p2])[::-1][:50].tolist()))
    cos     = cosine_similarity(shaps[p1], shaps[p2]).diagonal()
    results[(p1,p2)] = dict(rho=rho, pear=pear, ol50=ol50, cos=cos)
    print(f"  {p1} vs {p2} | {rho:>10.4f} | {pear:>8.4f} | "
          f"{ol50:>5d}/50 | {cos.mean():>10.4f} | {(cos>0.8).mean()*100:>6.1f}%")

## 4. Visualisation dashboard

In [ ]:
pair_cols = {('A','B'): '#D85A30', ('A','C'): '#1D9E75', ('B','C'): '#7F77DD'}
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Row 0: global importance scatters
for col, (p1,p2) in enumerate(pairs):
    ax = fig.add_subplot(gs[0,col])
    v  = results[(p1,p2)]
    ax.scatter(imps[p1], imps[p2], alpha=0.2, s=10,
               color=pair_cols[(p1,p2)], edgecolors='none')
    top20 = np.argsort(imps['A'])[-20:]
    ax.scatter(imps[p1][top20], imps[p2][top20], s=50,
               color='black', zorder=5, alpha=0.8)
    ax.set(xlabel=f'Importance {p1}', ylabel=f'Importance {p2}',
           title=f'{p1} vs {p2}\nρ={v["rho"]:.3f}  r={v["pear"]:.3f}')
    ax.grid(alpha=0.2)

# Row 0 col 3: summary bar
ax_bar = fig.add_subplot(gs[0,3])
x_ = np.arange(3); w = 0.35
rhos  = [results[p]['rho']  for p in pairs]
pears = [results[p]['pear'] for p in pairs]
ax_bar.bar(x_-w/2, rhos,  w, label='Spearman ρ', color='#534AB7', alpha=0.85)
ax_bar.bar(x_+w/2, pears, w, label='Pearson r',  color='#1D9E75', alpha=0.85)
ax_bar.set_xticks(x_); ax_bar.set_xticklabels(['A-B','A-C','B-C'], fontsize=9)
ax_bar.axhline(0.7, color='red', linestyle='--', lw=1, alpha=0.5)
ax_bar.set(title='Global importance agreement', ylabel='Correlation')
ax_bar.legend(fontsize=8); ax_bar.grid(alpha=0.3, axis='y')

# Row 1: cosine distributions
for col, (p1,p2) in enumerate(pairs):
    ax  = fig.add_subplot(gs[1,col])
    cos = results[(p1,p2)]['cos']
    ax.hist(cos, bins=50, color=pair_cols[(p1,p2)], edgecolor='black', alpha=0.8, density=True)
    ax.axvline(cos.mean(), color='black', lw=2, linestyle='--',
               label=f'mean={cos.mean():.3f}')
    ax.set(xlabel='Cosine similarity', ylabel='Density',
           title=f'{p1} vs {p2} — per-compound agreement')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Row 1 col 3: boxplot
ax_box = fig.add_subplot(gs[1,3])
bp = ax_box.boxplot([results[p]['cos'] for p in pairs],
                     labels=['A-B','A-C','B-C'],
                     patch_artist=True, medianprops={'color':'black','lw':2})
for patch, p in zip(bp['boxes'], pairs):
    patch.set_facecolor(pair_cols[p]); patch.set_alpha(0.7)
ax_box.axhline(0.5, color='red', linestyle='--', lw=1, alpha=0.5)
ax_box.set(title='Cosine similarity', ylabel='Cosine similarity')
ax_box.grid(alpha=0.3, axis='y')

# Row 2: top-30 importance bars
top30_A = np.argsort(imp_A)[::-1][:30]
for col, (label, imp_v, col_v) in enumerate([
    ('A (direct sel.)',    imp_A, '#1D9E75'),
    ('B (indiv. sub.)',   imp_B, '#D85A30'),
    ('C (multi-task sub.)',imp_C,'#7F77DD'),
]):
    ax = fig.add_subplot(gs[2,col])
    ax.barh(range(30), imp_v[top30_A], color=col_v, alpha=0.8)
    ax.set_yticks(range(30))
    ax.set_yticklabels([f'Bit {b}' for b in top30_A], fontsize=6)
    ax.set(xlabel='Mean |SHAP|', title=f'Top-30 bits (ranked by A)\nApproach {label}')
    ax.invert_yaxis(); ax.grid(alpha=0.3, axis='x')

# Row 2 col 3: top-N overlap matrix
ns = [10, 20, 30, 50, 100]
ol_m = np.zeros((3, len(ns)))
for i, (p1,p2) in enumerate(pairs):
    for j, n in enumerate(ns):
        s1 = set(np.argsort(imps[p1])[::-1][:n].tolist())
        s2 = set(np.argsort(imps[p2])[::-1][:n].tolist())
        ol_m[i,j] = len(s1&s2)/n*100
ax_ol = fig.add_subplot(gs[2,3])
im_ = ax_ol.imshow(ol_m, cmap='YlGn', vmin=0, vmax=100, aspect='auto')
ax_ol.set_xticks(range(len(ns))); ax_ol.set_xticklabels([f'Top-{n}' for n in ns], fontsize=8)
ax_ol.set_yticks(range(3)); ax_ol.set_yticklabels(['A-B','A-C','B-C'], fontsize=8)
for i in range(3):
    for j in range(len(ns)):
        ax_ol.text(j, i, f'{ol_m[i,j]:.0f}%', ha='center', va='center',
                   fontsize=9, color='black' if ol_m[i,j] < 70 else 'white')
plt.colorbar(im_, ax=ax_ol, label='% overlap')
ax_ol.set_title('Top-N overlap (%)', fontsize=10)

fig.suptitle('Three-way SHAP comparison: A · B · C in common 2048-bit space',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('fig_03_threeway.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. SHAP beeswarm — side by side

In [ ]:
top15_idx   = np.argsort(imp_A)[-15:][::-1]
feat_names  = [f'Bit {i}' for i in range(N_FP)]
top_names   = [feat_names[i] for i in top15_idx]
X_top_raw   = fps_shap[:, top15_idx]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, (label, sv) in zip(axes, [
    ('A — direct selectivity model', shap_A),
    ('B — individual model subtraction', shap_B),
    ('C — multi-task model subtraction', shap_C),
]):
    plt.sca(ax)
    shap.summary_plot(sv[:, top15_idx], X_top_raw,
                      feature_names=top_names, max_display=15,
                      show=False, plot_size=None)
    ax.set_title(f'Approach {label}', fontsize=11)

plt.suptitle('SHAP beeswarm — top-15 bits ranked by Approach A',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_03_beeswarm.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. Fragment mapping — chemical validation
Decode top Morgan bits to actual molecular substructures.

In [ ]:
TOP_N = 30
rank_A = np.argsort(imp_A)[::-1][:TOP_N]; set_A = set(rank_A.tolist())
rank_B = np.argsort(imp_B)[::-1][:TOP_N]; set_B = set(rank_B.tolist())
rank_C = np.argsort(imp_C)[::-1][:TOP_N]; set_C = set(rank_C.tolist())

shared_all  = set_A & set_B & set_C
unique_A    = set_A - set_B - set_C
unique_B    = set_B - set_A - set_C
unique_C    = set_C - set_A - set_B

print(f"Top-{TOP_N} feature membership:")
print(f"  Shared A∩B∩C: {len(shared_all)} bits")
print(f"  Unique to A:  {len(unique_A)} bits  ← selectivity-driving")
print(f"  Unique to B:  {len(unique_B)} bits  ← individual model artefacts")
print(f"  Unique to C:  {len(unique_C)} bits  ← multi-task artefacts")

# Overlap bar chart
groups = ['A∩B∩C','Only A','Only B','Only C']
counts = [len(shared_all), len(unique_A), len(unique_B), len(unique_C)]
cols   = ['#534AB7','#1D9E75','#D85A30','#7F77DD']
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(groups, counts, color=cols, alpha=0.85, edgecolor='black')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            str(cnt), ha='center', fontsize=12, fontweight='bold')
ax.set(title=f'Top-{TOP_N} feature membership across approaches', ylabel='Bits')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('fig_03_venn.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Fragment grids
draw_fragment_grid(unique_A,   imp_A, smiles_shap,
                   'Unique to A — selectivity-driving fragments',
                   save_path='fig_03_frags_A.png')

draw_fragment_grid(shared_all, imp_A, smiles_shap,
                   'Shared by all three approaches',
                   save_path='fig_03_frags_shared.png')

draw_fragment_grid(unique_B,   imp_B, smiles_shap,
                   'Unique to B — individual model artefacts',
                   save_path='fig_03_frags_B.png')

## 7. Compound-level divergence analysis

In [ ]:
cos_AB = cosine_similarity(shap_A, shap_B).diagonal()
cos_AC = cosine_similarity(shap_A, shap_C).diagonal()
cos_BC = cosine_similarity(shap_B, shap_C).diagonal()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (cos, label, col) in zip(axes, [
    (cos_AB, 'A vs B', '#D85A30'),
    (cos_AC, 'A vs C', '#1D9E75'),
    (cos_BC, 'B vs C', '#7F77DD'),
]):
    rho, _ = spearmanr(y_shap, cos)
    ax.scatter(y_shap, cos, alpha=0.25, s=12, color=col, edgecolors='none')
    z = np.polyfit(y_shap, cos, 1)
    x_line = np.linspace(y_shap.min(), y_shap.max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'k--', lw=2)
    ax.set(xlabel='ΔpChEMBL (atypicality)', ylabel='Cosine similarity',
           title=f'{label}\nSpearman ρ={rho:.3f}')
    ax.grid(alpha=0.2)

plt.suptitle('Per-compound SHAP divergence vs atypicality',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_03_divergence.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Summary

In [ ]:
print("=" * 65)
print("SELECTIVITY EXPLANATION SUMMARY")
print("=" * 65)
for (p1,p2), v in results.items():
    print(f"  {p1} vs {p2}: rho={v['rho']:.4f}  pearson={v['pear']:.4f}  "
          f"top50={v['ol50']}/50  cos={v['cos'].mean():.4f}  "
          f"%>0.8={(v['cos']>0.8).mean()*100:.1f}%")
print(f"\nUnique-to-A fragments (selectivity-driving): {len(unique_A)} bits")
print(f"Chemical validation: piperazine linker, cyclic amides, stereocentres")
print(f"Unique-to-B fragments (individual model artefacts): {len(unique_B)} bits")
print(f"Includes S-N (sulfonamide/5HT6 feature) — irrelevant to D2/5HT2A ratio")